# 03 — Biological interpretation

Uses **`results/top_cpgs.csv`** from notebook 02 (non-zero Elastic Net coefficients), merges Illumina 450k manifest annotations, compares against Horvath clock probes, runs a Fisher exact test on overlap versus the full array background (size from **`data/processed/n_cpgs.txt`**), performs GOterm enrichment via **g:Profiler**, and saves consolidated tables.

**Prerequisites:** run `01_data_loading.ipynb` (creates `n_cpgs.txt`) and `02_model_training.ipynb` (creates `top_cpgs.csv`).

## Dependencies

In [ ]:
%pip install -q pandas scipy matplotlib

## Paths

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Load model probes and array background size

`n_cpgs.txt` avoids reloading the full beta matrix for the Fisher exact background.

In [ ]:
import pandas as pd

top_path = RESULTS_DIR / "top_cpgs.csv"
n_cpgs_path = PROCESSED_DIR / "n_cpgs.txt"

if not top_path.is_file():
    raise FileNotFoundError(
        "Run 02_model_training.ipynb first. Missing:\n" + str(top_path)
    )
if not n_cpgs_path.is_file():
    raise FileNotFoundError(
        "Run 01_data_loading.ipynb first. Missing:\n" + str(n_cpgs_path)
    )

top_cpgs_df = pd.read_csv(top_path)
n_cpgs_background = int(n_cpgs_path.read_text(encoding="utf-8").strip())

top_cpgs_df = top_cpgs_df.rename(columns={"ID_REF": "cpg"})
my_cpgs = set(top_cpgs_df["cpg"].astype(str))

print("Non-zero model probes:", len(my_cpgs))
print("Array background CpGs:", n_cpgs_background)

## Illumina 450k manifest (download once, cache locally)

In [ ]:
annotation_url = (
    "https://webdata.illumina.com/downloads/productfiles/"
    "humanmethylation450/humanmethylation450_15017482_v1-2.csv"
)
annotation_path = RAW_DIR / "illumina450k_annotation.csv"

if not annotation_path.exists():
    annotation = pd.read_csv(annotation_url, skiprows=7, low_memory=False)
    annotation.to_csv(annotation_path, index=False)
else:
    annotation = pd.read_csv(annotation_path, low_memory=False)

print("Manifest shape:", annotation.shape)

## Merge annotations → `top_cpgs_annotated.csv`

In [ ]:
annotation_subset = annotation[[
    "IlmnID", "CHR", "MAPINFO",
    "UCSC_RefGene_Name", "UCSC_RefGene_Group",
    "Relation_to_UCSC_CpG_Island",
]].copy()

merged = top_cpgs_df.merge(
    annotation_subset,
    left_on="cpg",
    right_on="IlmnID",
    how="left",
)

annotated_path = RESULTS_DIR / "top_cpgs_annotated.csv"
merged.to_csv(annotated_path, index=False)
display(merged.head(15))

## Horvath clock probes (supplementary CSV)

Place **`13059_2013_3156_MOESM3_ESM.csv`** under `data/raw/` (Horvath et al., Genome Biology 2013 supplementary).

In [ ]:
import urllib.request

horvath_csv = RAW_DIR / "13059_2013_3156_MOESM3_ESM.csv"

if not horvath_csv.is_file():
    url = (
        "https://static-content.springer.com/esm/"
        "art%3A10.1186%2Fgb-2013-14-10-r115/"
        "MediaObjects/13059_2013_3156_MOESM3_ESM.csv"
    )

    print("Downloading Horvath supplementary CSV...")
    urllib.request.urlretrieve(url, horvath_csv)
    print("Saved to:", horvath_csv)

horvath_raw = pd.read_csv(horvath_csv, header=None)

new_columns = horvath_raw.iloc[2].tolist()

horvath = horvath_raw.iloc[3:].copy()
horvath.columns = new_columns
horvath = horvath.reset_index(drop=True)

horvath = horvath[
    horvath["CpGmarker"] != "(Intercept)"
].copy()

horvath_cpgs = set(
    horvath["CpGmarker"].astype(str)
)

print("Horvath probes:", len(horvath_cpgs))

## Overlap table → `horvath_overlap.csv`

In [ ]:
overlap = my_cpgs & horvath_cpgs
print("Overlap size:", len(overlap))

overlap_df = merged[merged["cpg"].isin(overlap)].copy()
overlap_df = overlap_df.merge(
    horvath[[
        "CpGmarker", "CoefficientTraining", "CoefficientTrainingShrunk",
        "Symbol", "Marginal Age Relationship",
    ]],
    left_on="cpg",
    right_on="CpGmarker",
    how="left",
)
overlap_df = overlap_df.sort_values("abs_coef", ascending=False)

horvath_overlap_path = RESULTS_DIR / "horvath_overlap.csv"
overlap_df.to_csv(horvath_overlap_path, index=False)

display(
    overlap_df[[
        "cpg", "coefficient", "abs_coef", "UCSC_RefGene_Name", "Symbol",
        "CoefficientTraining", "CoefficientTrainingShrunk",
        "Marginal Age Relationship",
    ]].head(20)
)

## Fisher exact test (overlap enrichment vs array background)

In [ ]:
from scipy.stats import fisher_exact

a = len(overlap)
b = len(my_cpgs - horvath_cpgs)
c = len(horvath_cpgs - my_cpgs)
d = n_cpgs_background - len(my_cpgs | horvath_cpgs)

table = [[a, b], [c, d]]
odds_ratio, p_value = fisher_exact(table, alternative="greater")

print("Contingency table [[overlap, model_only],[horvath_only, neither]]:")
print(table)
print("Odds ratio:", odds_ratio)
print("One-sided P (greater overlap):", p_value)

## GO term enrichment (g:Profiler REST)

Gene symbols are parsed from **`UCSC_RefGene_Name`** (same convention as the training notebook).

In [ ]:
import json
import urllib.request


def refgene_to_symbols(refgene):
    if pd.isna(refgene) or refgene == "":
        return []
    symbols = []
    for chunk in str(refgene).split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        base = chunk.split("(")[0].strip()
        if base:
            symbols.append(base)
    return symbols


genes = []
for val in merged["UCSC_RefGene_Name"]:
    genes.extend(refgene_to_symbols(val))

query_genes = sorted(set(genes))
print("Unique gene symbols for enrichment:", len(query_genes))


def run_gprofiler(query, organism="hsapiens"):
    url = "https://biit.cs.ut.ee/gprofiler/api/gost/profile/"
    payload = {
        "query": query,
        "organism": organism,
        "sources": ["GO:BP", "GO:MF", "GO:CC"],
        "user_threshold": 0.05,
        "significance_threshold_method": "g_SCS",
        "all_results": False,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read().decode("utf-8"))


go_rows = []
if query_genes:
    gprof = run_gprofiler(query_genes)
    for item in gprof.get("result", []) or []:
        go_rows.append(
            {
                "analysis_type": "gprofiler_go",
                "native_id": item.get("native"),
                "term_name": item.get("name"),
                "source": item.get("source"),
                "p_value": item.get("p_value"),
                "term_size": item.get("term_size"),
                "query_size": item.get("query_size"),
                "intersection_size": item.get("intersection_size"),
                "recall": item.get("recall"),
                "precision": item.get("precision"),
            }
        )
else:
    print("No genes parsed from annotation; skipping g:Profiler.")

go_df = pd.DataFrame(go_rows)
if not go_df.empty:
    go_df = go_df.sort_values(["source", "p_value", "native_id"])
display(go_df.head(20))

## Combine Fisher summary + GO results → `enrichment_results.csv`

In [ ]:
fisher_df = pd.DataFrame(
    [
        {
            "analysis_type": "fisher_exact_horvath_overlap",
            "native_id": "FISHER",
            "term_name": "CpG overlap vs expected (model ∩ Horvath)",
            "source": "overlap_test",
            "p_value": float(p_value),
            "odds_ratio": float(odds_ratio),
            "a_overlap": int(a),
            "b_model_only": int(b),
            "c_horvath_only": int(c),
            "d_neither": int(d),
            "n_my_cpgs": len(my_cpgs),
            "n_horvath_cpgs": len(horvath_cpgs),
            "n_background_cpgs": int(n_cpgs_background),
        }
    ]
)

if go_df.empty:
    enrichment_all = fisher_df.copy()
else:
    enrichment_all = pd.concat([fisher_df, go_df], ignore_index=True, sort=False)

enrichment_path = RESULTS_DIR / "enrichment_results.csv"
enrichment_all.to_csv(enrichment_path, index=False)

## Biological plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not go_df.empty:
    plot_df = go_df.dropna(subset=["p_value"]).copy()
    plot_df = plot_df.nsmallest(20, "p_value")
    plot_df["neglog10_p"] = -np.log10(plot_df["p_value"].clip(lower=1e-300))

    fig, ax = plt.subplots(figsize=(10, 8))
    order = plot_df.sort_values("neglog10_p")
    labels = [
        f"{row['native_id']} — {str(row['term_name'])[:60]}"
        for _, row in order.iterrows()
    ]
    ax.barh(labels, order["neglog10_p"])
    ax.set_xlabel("-log10(P)")
    ax.set_title("Top GO terms (g:Profiler)")
    fig.tight_layout()
    plt.show()

chr_counts = merged["CHR"].astype(str).value_counts().sort_index()
fig2, ax2 = plt.subplots(figsize=(10, 4))
chr_counts.plot(kind="bar", ax=ax2)
ax2.set_xlabel("Chromosome")
ax2.set_ylabel("Non-zero model CpGs")
ax2.set_title("Chromosomal distribution (annotated probes)")
fig2.tight_layout()
plt.show()

## Export summary

In [ ]:
exports = [
    annotated_path,
    horvath_overlap_path,
    enrichment_path,
]
print("Exported:")
for p in exports:
    print(" -", p.resolve())